In [10]:
import json
import pandas as pd
from collections import defaultdict

# Model name mapping
model_mapping = {
    "gpt-4.0-mini": "GPT-4o",
    "anthropic.claude-3-5-sonnet-20241022-v2:0": "CL-3.5", 
    "gemini-1.5-flash-002": "Gem-1.5",
    "Llama-3.1-8B-Instruct": "L-3.1",
    "Qwen2.5-14B-Instruct-1M": "Q-2.5",
    "Qwen2.5-Coder-7B-Instruct": "Qwen2.5-C",
    "DeepSeek-Coder-V2-Lite-Instruct": "DS-C",
    "Phi-4-mini-instruct": "Phi-4",
    "Phi-3.5-mini-instruct": "Phi-3.5",
    "granite-3.2-8b-instruct": "Gr-3.2",
    "DeepSeek-R1-Distill-Qwen-7B": "DSR1-Q-7B",
    "DeepSeek-R1-Distill-Llama-8B": "DSR1-L", 
    "DeepSeek-R1-Distill-Qwen-14B": "DSR1-Q-14B",
    "granite-3.2-8b-instruct-preview": "Granite-3.2 Pr"
}

# Column name mapping
column_mapping = {
    "Assignment": "Variable Assignment",
    "API": "API Call", 
    "Branch": "Boolean",
    "Constant Assignment": "Constant",
    "Arithmetic Assignment": "Arithmetic"
}

# Read the JSONL file
file_path = "/home/XXX/CodeSemantic/CodeSemantic/statement_Accuracy_Results_ICLR/statement_python.jsonl"
data = []

with open(file_path, 'r') as file:
    for line in file:
        data.append(json.loads(line.strip()))

# Filter for shot=2 and API definition=no
filtered_data = [entry for entry in data if entry.get('shot') == 2 and entry.get('API definition') == 'no']

# Organize data by model and type accuracy
model_accuracy = defaultdict(dict)

for entry in filtered_data:
    model_full_name = entry['Model']
    # Map to shorter ID, fallback to original name if not found
    model_id = model_mapping.get(model_full_name, model_full_name)
    type_acc = entry['type_accuracy']
    
    for accuracy_type, accuracy_value in type_acc.items():
        # Map column name, fallback to original if not found
        mapped_column = column_mapping.get(accuracy_type, accuracy_type)
        model_accuracy[mapped_column][model_id] = accuracy_value

# Convert to DataFrame
df = pd.DataFrame(model_accuracy).T

# Format the accuracy values as percentages with 2 decimal places
df = df.round(2)

# Save to Excel
output_file = "statement_accuracy_results.xlsx"
df.to_excel(output_file)

print(f"Excel file saved as: {output_file}")
print("\nData preview:")
print(df)

Excel file saved as: statement_accuracy_results.xlsx

Data preview:
                     Q-2.5  Qwen2.5-C  DS-C  Phi-4  Phi-3.5  Gr-3.2  \
Variable Assignment   0.68       0.67  0.68   0.56     0.55    0.59   
Boolean               0.79       0.72  0.72   0.57     0.60    0.74   
API Call              0.24       0.22  0.17   0.13     0.12    0.13   
Arithmetic            0.54       0.46  0.44   0.32     0.35    0.33   
Constant              0.91       0.90  0.84   0.90     0.88    0.93   

                     DSR1-Q-7B  DSR1-L  DSR1-Q-14B  Granite-3.2 Pr  CL-3.5  \
Variable Assignment       0.55    0.56        0.68            0.62    0.79   
Boolean                   0.47    0.56        0.76            0.74    0.80   
API Call                  0.08    0.12        0.22            0.13    0.34   
Arithmetic                0.42    0.42        0.54            0.29    0.68   
Constant                  0.82    0.66        0.85            0.91    0.93   

                     gpt-4o-mini  
V

In [13]:
import json
import pandas as pd
from collections import defaultdict

# Read the JSONL file
file_path = "/home/XXX/CodeSemantic/CodeSemantic/statement_Accuracy_Results_ICLR/statement_python.jsonl"

# Store data for comparison
comparison_data = defaultdict(dict)

with open(file_path, 'r') as file:
    for line in file:
        data = json.loads(line.strip())
        
        # Filter for shot = 2
        if data.get('shot') == 0:
            model = data['Model']
            api_def = data.get('API definition', 'no')
            accuracy = data.get('accuracy', 0)
            
            # Store accuracy based on API definition type
            if api_def == 'no':
                comparison_data[model]['api_definition_no'] = accuracy
            elif api_def == 'code':
                comparison_data[model]['api_definition_code'] = accuracy

# Convert to DataFrame
df_comparison = pd.DataFrame.from_dict(comparison_data, orient='index')

# Reset index to make Model a column
df_comparison.reset_index(inplace=True)
df_comparison.rename(columns={'index': 'Model'}, inplace=True)

# Reorder columns
df_comparison = df_comparison[['Model', 'api_definition_no', 'api_definition_code']]

# Fill NaN values with a placeholder (e.g., 'N/A') if any models don't have both conditions
df_comparison.fillna('N/A', inplace=True)

# Save to Excel
output_path = "api_definition_comparison_shot2.xlsx"
df_comparison.to_excel(output_path, index=False)

print("Comparison Excel file created successfully!")
print(f"File saved at: {output_path}")
print("\nData preview:")
print(df_comparison)

Comparison Excel file created successfully!
File saved at: api_definition_comparison_shot2.xlsx

Data preview:
                              Model api_definition_no  api_definition_code
0             Llama-3.1-8B-Instruct               N/A             0.313761
1           Qwen2.5-14B-Instruct-1M          0.433028             0.436697
2         Qwen2.5-Coder-7B-Instruct          0.383486             0.365138
3   DeepSeek-Coder-V2-Lite-Instruct          0.293578             0.299083
4               Phi-4-mini-instruct          0.348624             0.335780
5             Phi-3.5-mini-instruct          0.266055             0.266055
6           granite-3.2-8b-instruct          0.194495             0.203670
7       DeepSeek-R1-Distill-Qwen-7B          0.225688             0.236697
8      DeepSeek-R1-Distill-Llama-8B          0.306422             0.266055
9      DeepSeek-R1-Distill-Qwen-14B          0.398165             0.401835
10  granite-3.2-8b-instruct-preview          0.194495           

/tmp/ipykernel_2441855/3015518997.py:38: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_comparison.fillna('N/A', inplace=True)
